<a href="https://colab.research.google.com/github/Purva1204/DL_LAB_assignments/blob/main/Assignment-8/assignment8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate scikit-learn

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [3]:
data = {
    "text": [
        "I loved this movie, it was amazing!",
        "The product quality is excellent.",
        "I really enjoyed the experience.",
        "This is the best service I have ever received.",
        "The food was delicious and fresh.",
        "I am very happy with my purchase.",
        "The movie was fantastic and entertaining.",
        "The service was quick and friendly.",
        "I hated this movie, it was terrible.",
        "The product quality is very poor.",
        "I did not enjoy the experience.",
        "This is the worst service ever.",
        "The food was cold and tasteless.",
        "I am disappointed with my purchase.",
        "The movie was boring and terrible.",
        "The service was slow and rude."
    ],

    "label": [
        1, 1, 1, 1, 1, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0
    ]
}

df = pd.DataFrame(data)

df

,text,label
0,"I loved this movie, it was amazing!",1
1,The product quality is excellent.,1
2,I really enjoyed the experience.,1
3,This is the best service I have ever received.,1
4,The food was delicious and fresh.,1
5,I am very happy with my purchase.,1
6,The movie was fantastic and entertaining.,1
7,The service was quick and friendly.,1
8,"I hated this movie, it was terrible.",0
9,The product quality is very poor.,0


In [4]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.25,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(train_texts))
print("Testing samples:", len(test_texts))

Training samples: 12
Testing samples: 4


In [5]:
model_name = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization completed!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenization completed!


In [6]:
class SentimentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = SentimentDataset(
    train_encodings,
    train_labels
)

test_dataset = SentimentDataset(
    test_encodings,
    test_labels
)

print("Dataset ready!")

Dataset ready!


In [7]:
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("BERT model loaded successfully!")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded successfully!


In [8]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy
    }

In [9]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=5,
    report_to="none"
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,0.686704


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9, training_loss=0.6786449220445421, metrics={'train_runtime': 33.6778, 'train_samples_per_second': 1.069, 'train_steps_per_second': 0.267, 'total_flos': 221999952960.0, 'train_loss': 0.6786449220445421, 'epoch': 3.0})

In [11]:
results = trainer.evaluate()

print("Evaluation Results:")
print(results)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy
0.686704,0.628994,9,0.750000


Evaluation Results:
{'eval_loss': 0.6289941668510437, 'eval_accuracy': 0.75}


In [12]:
predictions = trainer.predict(test_dataset)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=1
)

print(classification_report(
    test_labels,
    predicted_labels,
    target_names=["Negative", "Positive"]
))

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

    Negative       0.67      1.00      0.80         2
    Positive       1.00      0.50      0.67         2

    accuracy                           0.75         4
   macro avg       0.83      0.75      0.73         4
weighted avg       0.83      0.75      0.73         4



In [13]:
def predict_sentiment(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    device = model.device
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    if prediction == 1:
        return "Positive"
    else:
        return "Negative"

In [14]:
def predict_sentiment(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    device = model.device
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    if prediction == 1:
        return "Positive"
    else:
        return "Negative"

In [15]:
sentences = [
    "I really enjoyed this product.",
    "The experience was horrible.",
    "The service was excellent.",
    "I am very disappointed.",
    "This movie was fantastic."
]

for sentence in sentences:
    print(sentence)
    print("Sentiment:", predict_sentiment(sentence))
    print()

I really enjoyed this product.
Sentiment: Negative

The experience was horrible.
Sentiment: Negative

The service was excellent.
Sentiment: Negative

I am very disappointed.
Sentiment: Negative

This movie was fantastic.
Sentiment: Positive

